In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    brier_score_loss,
)
import json

from sklearn.calibration import calibration_curve

from leadgate.pipeline import make_champion_pipeline, make_cv
from leadgate.threshold import pick_best, threshold_sweep

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

In [ ]:
cv = make_cv()
log_reg_pipe = make_champion_pipeline()

In [ ]:
proba = cross_val_predict(
    log_reg_pipe, X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]
print(proba.shape)

In [ ]:
precision, recall, threshold = precision_recall_curve(y_train, proba)
ap = average_precision_score(y_train, proba)
plt.plot(recall, precision, label=f"PR-AUC (AP) = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve")
plt.legend()
plt.show()

To make this project more interesting I will introduce two variables which will impact decision making.

- `r` = revenue from a successful call = $100.
- `c` = cost of the call = $10.

Also our call-center can only call 2000 customers a day because we have a little team of callers.

In [ ]:
r = 100
c = 10

df = threshold_sweep(y_train, proba, revenue=r, cost=c)
best_threshold_nolimit = pick_best(df)
print(f"The best threshold for unlimited calls: {best_threshold_nolimit['threshold']}")
best_threshold_limit = pick_best(df, max_calls=2000)
print(f"The best threshold for 2000 calls: {best_threshold_limit['threshold']}")

The best economic situation is on a threshold of 0.11, but in this scenario we need to make 12k calls a day. Otherwise, the most optimal threshold which our call-center can use is 0.36 with 2k calls.

In [ ]:
with open("../models/threshold.json", "w") as f:
    json.dump(best_threshold_nolimit.to_dict(), f, indent=2)

Now I want to check if our model is calibrated and thresholds do not lie to us.

In [ ]:
prob_true, prob_pred = calibration_curve(y_train, proba, strategy="quantile")
plt.plot(prob_true, prob_pred, marker="o", label="model")
plt.plot([0, 1], [0, 1], "--", label="ideal")
plt.title("Calibration Curve (quantile bins, n=5)")
plt.ylabel("predicted probability")
plt.xlabel("Actual proportion of class 1")
plt.legend()
print(brier_score_loss(y_train, proba))
plt.savefig("../reports/figures/calibration.png")
plt.show()

We can see that the Brier score is 0.086, which is good, and our model is normally calibrated. We can see that the model does not give high probabilities, it is caused by the imbalance of classes.